### Gold Layer (DLT)
Materialized Views for the dashboard. These recompute fully on each pipeline update (not incremental), because the 7/30/90-day trend windows need to look back across the whole accumulated `silver_quotes` history, not just newly-arrived rows.

Note: with the quote endpoint's daily snapshot cadence, the 90-day window naturally fills in as the pipeline runs daily over time — the logic is correct from day one, it just returns `NULL` for the longer windows until enough history has accumulated.

In [0]:
import dlt
from pyspark.sql import functions as F
from pyspark.sql.window import Window

**Price Trend Analysis** - daily price change and % change over 7/30/90 days

In [0]:
@dlt.table(
    name="gold_price_trend",
    comment="Daily price change and pct change over 7, 30, and 90 day lookback windows, per symbol.",
    table_properties={"quality": "gold"}
)
def gold_price_trend():
    df = dlt.read("silver_quotes")
    w = Window.partitionBy("symbol").orderBy("trading_date")

    return (
        df
        .withColumn("price_7d_ago", F.lag("price", 7).over(w))
        .withColumn("price_30d_ago", F.lag("price", 30).over(w))
        .withColumn("price_90d_ago", F.lag("price", 90).over(w))
        .withColumn("price_change_7d", F.col("price") - F.col("price_7d_ago"))
        .withColumn("price_change_30d", F.col("price") - F.col("price_30d_ago"))
        .withColumn("price_change_90d", F.col("price") - F.col("price_90d_ago"))
        .withColumn("price_pct_change_7d", F.col("price_change_7d") / F.col("price_7d_ago") * 100)
        .withColumn("price_pct_change_30d", F.col("price_change_30d") / F.col("price_30d_ago") * 100)
        .withColumn("price_pct_change_90d", F.col("price_change_90d") / F.col("price_90d_ago") * 100)
        .select(
            "symbol", "trading_date", "price",
            "price_change_7d", "price_pct_change_7d",
            "price_change_30d", "price_pct_change_30d",
            "price_change_90d", "price_pct_change_90d"
        )
    )

**Volume Trend Analysis** - rolling average daily volume over 7/30/90 days

In [0]:
@dlt.table(
    name="gold_volume_trend",
    comment="Rolling average trading volume over 7, 30, and 90 day windows, per symbol.",
    table_properties={"quality": "gold"}
)
def gold_volume_trend():
    df = dlt.read("silver_quotes")
    w7 = Window.partitionBy("symbol").orderBy("trading_date").rowsBetween(-6, 0)
    w30 = Window.partitionBy("symbol").orderBy("trading_date").rowsBetween(-29, 0)
    w90 = Window.partitionBy("symbol").orderBy("trading_date").rowsBetween(-89, 0)

    return (
        df
        .withColumn("avg_volume_7d", F.avg("volume").over(w7))
        .withColumn("avg_volume_30d", F.avg("volume").over(w30))
        .withColumn("avg_volume_90d", F.avg("volume").over(w90))
        .select("symbol", "trading_date", "volume", "avg_volume_7d", "avg_volume_30d", "avg_volume_90d")
    )

**Ticker Snapshot** - convenience view joining latest quote + current company profile, the natural source table for a Yahoo-Finance-style overview card.

In [0]:
@dlt.table(
    name="gold_ticker_snapshot",
    comment="Latest quote joined with current company profile - one row per symbol, dashboard-ready.",
    table_properties={"quality": "gold"}
)
def gold_ticker_snapshot():
    quotes = dlt.read("silver_quotes")
    company = dlt.read("silver_company_info").filter(F.col("__END_AT").isNull())  # current SCD2 record only

    latest_quote_window = Window.partitionBy("symbol").orderBy(F.desc("trading_date"))
    latest_quote = (
        quotes
        .withColumn("rn", F.row_number().over(latest_quote_window))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )

    return (
        latest_quote
        .join(company, on="symbol", how="left")
        .select(
            latest_quote["symbol"], "company_name", "sector", "industry", "exchange",
            "price", "day_change", "day_change_pct", "day_high", "day_low", "volume",
            "market_cap", "pe_ratio", "week_52_high", "week_52_low", "trading_date"
        )
    )